# Lesson 2: Building Blocks of CNNs

**Duration:** 60 minutes

## Learning Objectives
- Understand convolution operations and their purpose
- Learn about pooling layers, activation functions, and network architecture
- Build a simple CNN from scratch using PyTorch
- Understand how data flows through a network

## 2.1 Understanding Convolution

**Convolution** applies a small filter (kernel) across the image to extract local features.
- Kernel size: typically 3x3, 5x5
- Stride: how many pixels to move the kernel
- Padding: adding zeros around the image
- Output: feature map

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Create a simple image
image = torch.randn(1, 1, 5, 5)  # Batch, Channels, Height, Width
print(f'Input image shape: {image.shape}')
print(f'Input image:\n{image.squeeze()}')

# Create a convolution layer
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0)
print(f'\nConvolution kernel shape: {conv.weight.shape}')

# Apply convolution
output = conv(image)
print(f'\nOutput shape after Conv2d(kernel_size=3, stride=1, padding=0): {output.shape}')
print(f'Output:\n{output.squeeze()}')

## 2.2 Pooling Layers

**Pooling** reduces spatial dimensions by downsampling the feature maps.
- **Max Pooling:** Keep the maximum value in a window
- **Average Pooling:** Compute the average in a window
- Reduces computation and helps with translation invariance

In [ ]:
# Max Pooling
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
pooled_output = max_pool(output)
print(f'Output shape after MaxPool2d(kernel_size=2): {pooled_output.shape}')
print(f'Pooled output:\n{pooled_output.squeeze()}')

# Average Pooling
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
avg_pooled = avg_pool(output)
print(f'\nOutput shape after AvgPool2d(kernel_size=2): {avg_pooled.shape}')

## 2.3 Activation Functions

**Activation functions** introduce non-linearity to the network.
- **ReLU (Rectified Linear Unit):** $f(x) = max(0, x)$ - most common
- **Sigmoid:** $f(x) = \frac{1}{1+e^{-x}}$ - outputs between 0 and 1
- **Tanh:** $f(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ - outputs between -1 and 1

In [ ]:
# Visualize activation functions
x = torch.linspace(-5, 5, 100)
relu_out = F.relu(x)
sigmoid_out = torch.sigmoid(x)
tanh_out = torch.tanh(x)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].plot(x.numpy(), relu_out.numpy())
axes[0].set_title('ReLU')
axes[0].grid(True)

axes[1].plot(x.numpy(), sigmoid_out.numpy())
axes[1].set_title('Sigmoid')
axes[1].grid(True)

axes[2].plot(x.numpy(), tanh_out.numpy())
axes[2].set_title('Tanh')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 2.4 Building a Simple CNN

Let's build a simple CNN from scratch:

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)  # 32x32 -> 32x32
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)  # 16x16 -> 16x16
        # Pool again: 16x16 -> 8x8
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        # Conv block 2
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layers
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        
        return x

# Initialize model
model = SimpleCNN(num_classes=10)
print(model)

## 2.5 Forward Pass Visualization

Let's trace how data flows through the network:

In [ ]:
# Test forward pass
batch_size = 4
input_tensor = torch.randn(batch_size, 3, 32, 32)  # Batch of 4 RGB images 32x32
print(f'Input shape: {input_tensor.shape}')

output = model(input_tensor)
print(f'Output shape: {output.shape}')
print(f'\nOutput (logits for 10 classes):\n{output}')

# Get class predictions
class_predictions = torch.argmax(output, dim=1)
print(f'\nPredicted classes: {class_predictions}')

# Get probabilities
probabilities = F.softmax(output, dim=1)
print(f'\nClass probabilities for first sample:\n{probabilities[0]}')
print(f'Max probability: {probabilities[0].max().item():.4f}')